In [0]:
dbutils.widgets.text("source_system","")
dbutils.widgets.text("table_name","")
dbutils.widgets.text("landing_timestamp","")
dbutils.widgets.text("source_file_format", "")

source_system=dbutils.widgets.get("source_system")
table_name=dbutils.widgets.get("table_name")
landing_timestamp=dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")



In [0]:
landing_path = f"/Volumes/adb_medisphere/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

try:
    files = dbutils.fs.ls(landing_path)

    if not files:
        dbutils.notebook.exit("No new files found")

except Exception:
    dbutils.notebook.exit("No new files found")

if source_file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    landing_df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql.functions import *

updated_df=landing_df.withColumn("landing_timestamp",lit(landing_timestamp))\
        .withColumn("insert_timestamp",current_timestamp())
    


In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_medisphere.bronze.{table_name}")

In [0]:
%sql
select * from adb_medisphere.bronze.hospitals